Thoughts:
- llama and command r can decently simplify sentences zero-shot.
- add HSK vocabulary context so simplification includes more HSK1-4 vocab.
- add user-specific vocab knowledge which gets added to context. This will be complex words that the user knows (don't simplify) or complex words the user doesn't know (if it is a key word to the meaning, keep it, otherwise, simplify it)
- optional end-goals/simplification level: (1) extensive reading, keep it as simple as possible (2) a little intensive so the user can learn more words

Notebook for experimenting with bedrock LLM for text simplification

In [1]:
import boto3
from botocore.exceptions import ClientError
from langchain_aws.llms.bedrock import BedrockLLM
from langchain_aws.chat_models import ChatBedrockConverse

In [72]:
# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID
arn_cr = "cohere.command-r-v1:0"
arn_ds = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.deepseek.r1-v1:0"
arn_llama = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-2-3b-instruct-v1:0"
arn_llama2 = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

llm_llama = ChatBedrockConverse(client=brt,
                        model_id=arn_llama,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=150
                        ,)

llm_llama2 = ChatBedrockConverse(client=brt,
                        model_id=arn_llama2,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=2000
                        ,)

llm_cr = ChatBedrockConverse(client=brt,
                        model=arn_cr,
                        temperature=0.1,
                        max_tokens=50,)

In [99]:
def sentence_pipeline(sentence):
    simplified = llm_llama.invoke(f"请简化下面的中文句子: {sentence}").content
    translated = llm_llama.invoke(f"英文翻译下面的句子: {simplified}").content
    return (sentence, simplified, translated)

In [146]:
import requests
import re
from bs4 import BeautifulSoup
import string
# from utils.TS_pipeline import TS_with_BART
from utils.bedrock_pipeline import sentence_pipeline
url = 'https://read.douban.com/reader/column/68275019/chapter/487097904/?dcs=column&dcm=chapter-list'
site = requests.get(url)
site.encoding = 'utf-8'
site_soup = BeautifulSoup(site.text, 'html.parser')
# print(site_soup.prettify())

In [118]:
def is_chinese(text: str, threshold=0.5) -> bool:
    # checks if a string is mostly Chinese characters
    chinese_chars = re.findall(r'[\u4e00-\u9fff]', text)
    return len(chinese_chars) / max(len(text), 1) >= threshold

In [143]:
sentences = []
html = []

for element in site_soup.find_all('p'):
    # print("Next:", element.decode_contents())
    paragraph = element.get_text(separator='', strip=True)  # Extract plain text without tags
    text = re.split(r'(?<=[。！？!?．])\s*', paragraph)
    sentences.extend([s.strip() for s in text if s.split()])

In [144]:
sentences

[]

In [ ]:
'''请简化下面这段中文，使每个句子的用词更简单，适合中文学习者阅读。请特别注意以下几点：

1. 替换生僻词、高级词汇或抽象表达，使用更常见、更直白的词语或短语。
2. 避免使用超出HSK 6级范围的词汇，优先使用HSK 1-5级中常见词汇。
3. 保留原句数量和顺序，只进行词语层面的简化，不要省略、合并或总结内容。
4. 只输出简化后的句子，不要解释或分析。'''

In [138]:
def build_prompt(sentences):
    prompt = ""
    base_instruction = '''请简化下面这段中文，使每个句子的用词更简单，适合中文学习者阅读。请特别注意以下几点：

1. 替换生僻词、高级词汇或抽象表达，使用更常见、更直白的词语或短语。
2. 避免使用超出HSK 6级范围的词汇，优先使用HSK 1-5级中常见词汇。
3. 保留原句数量和顺序，只进行词语层面的简化，不要省略、合并或总结内容。
4. 只输出简化后的句子，不要解释或分析。
5. 确保简化句子的语义与原句子保持一致。

原文段落：\n'''
    prompt += base_instruction

    for i, sentence in enumerate(sentences):
        prompt+=f"{i}. {sentence}\n"

    prompt+="\n请简化后的段落：\n"
    
    return prompt

In [139]:
prompt_sentences = build_prompt(sentences)
print(prompt_sentences)

请简化下面这段中文，使每个句子的用词更简单，适合中文学习者阅读。请特别注意以下几点：

1. 替换生僻词、高级词汇或抽象表达，使用更常见、更直白的词语或短语。
2. 避免使用超出HSK 6级范围的词汇，优先使用HSK 1-5级中常见词汇。
3. 保留原句数量和顺序，只进行词语层面的简化，不要省略、合并或总结内容。
4. 只输出简化后的句子，不要解释或分析。
5. 确保简化句子的语义与原句子保持一致。

原文段落：
0. 6月28日，重庆市永川区双石镇双石社区工作人员姜春艳通过手机上的“一表通”系统更新社区居民个人最新信息后，向其本人再次核实、确认。
1. 为深入贯彻中央八项规定精神、切实整治形式主义为基层减负，重庆市下大力气规范基层填表报数、推进“一表通”建设，依托全市统建的一体化智能化公共数据平台开展报表需求梳理工作，按照去重、合并、转换、编目、审核、上线“六步工作法”，乡镇（街道）报表数量和数据项都实现了不同程度的压减，帮助广大基层干部卸下“包袱”、轻装上阵，并促进数据资源共享，提高了行政效率。
2. 新华社记者 陈诚 摄
3. 分享让更多人看到
4. 人民日报社概况|关于人民网|报社招聘|招聘英才|广告服务|合作加盟|版权服务|数据服务|网站声明|网站律师|信息保护|联系我们
5. 人民日报违法和不良信息举报电话：010-65363263    举报邮箱：jubao@people.cn
6. 人民网服务邮箱：kf@people.cn违法和不良信息举报电话：010-65363636    举报邮箱：rmwjubao@people.cn
7. 互联网新闻信息服务许可证10120170001|增值电信业务经营许可证B1-20060139|广播电视节目制作经营许可证（广媒）字第172号|京ICP备12004265号-13
8. 信息网络传播视听节目许可证0104065|网络文化经营许可证 京网文[2023]4961-141号|网络出版服务许可证（京）字121号|京ICP证000006号|京公网安备11000002000008号
9. 人 民 网 股 份 有 限 公 司 版 权 所 有 ，未 经 书 面 授 权 禁 止 使 用Copyright © 1997-2025 by www.people.com.cn. all rights reserved

请简化后的段落：

In [140]:
response = llm_llama2.invoke(prompt_sentences).content

In [141]:
print(response)

0. 6月28日，重庆市永川区双石镇双石社区工作人员姜春艳通过手机上的系统更新社区居民个人最新信息后，向居民本人再次核实、确认。
1. 重庆市为了减轻基层工作人员的负担，规范基层填表报数，推进“一表通”建设，利用全市统一的数据平台开展报表工作，按照六步工作法，减少了乡镇（街道）报表数量和数据项，帮助基层干部减轻负担，提高了行政效率。
2. 新华社记者 陈诚 摄
3. 分享让更多人看到
4. 人民日报社概况|关于人民网|报社招聘|招聘英才|广告服务|合作加盟|版权服务|数据服务|网站声明|网站律师|信息保护|联系我们
5. 人民日报违法和不良信息举报电话：010-65363263    举报邮箱：jubao@people.cn
6. 人民网服务邮箱：kf@people.cn违法和不良信息举报电话：010-65363636    举报邮箱：rmwjubao@people.cn
7. 互联网新闻信息服务许可证10120170001|增值电信业务经营许可证B1-20060139|广播电视节目制作经营许可证（广媒）字第172号|京ICP备12004265号-13
8. 信息网络传播视听节目许可证0104065|网络文化经营许可证 京网文[2023]4961-141号|网络出版服务许可证（京）字121号|京ICP证000006号|京公网安备11000002000008号
9. 人民网股份有限公司版权所有，未经书面授权禁止使用Copyright 1997-2025 by www.people.com.cn. all rights reserved


In [124]:
def parse_llm_output(response):
    # split into lines
    lines = response.strip().split('\n')

    # remove numeric prefixes
    re_lines = [re.sub(r'^\d+\.\s*', '', line).strip() for line in lines if line.strip()]
    return re_lines

In [126]:
response_parsed = parse_llm_output(response)

In [130]:
print(response_parsed)

['6月30日，北京到阿塞拜疆巴库的火车停在北京房山区的北京国际陆港，等待出发。', '新华社记者 张晨霖 摄', '6月30日，北京的第一趟火车从房山区的北京西南金港物流基地出发。', '104个货箱装着2300多吨、价值1500多万元的汽车零件、机械设备、书籍等货物，开往阿塞拜疆首都巴库。', '6月30日，第一趟北京到阿塞拜疆巴库的火车出发。', '新华社记者 张晨霖 摄', '这趟火车采用“火车—船—火车”方式，将行驶8000多公里。', '火车从北京出发后，经过新疆霍尔果斯口岸出境，穿过哈萨克斯坦到达里海东岸的阿克套港，转乘船过里海，到阿利亚特港上岸，再由火车到巴库。', '之后，部分货物还将运往格鲁吉亚、土耳其、塞尔维亚等国家。', '6月30日，第一趟北京到阿塞拜疆巴库的火车出发。', '新华社记者 张晨霖 摄', '“这趟火车运行时间为15天左右，而原来的海运需要约50天。', '”火车组织者北京房山国际陆港运营有限公司总经理王传蒙说，通过铁路、船多式联运，不仅可以缩短运输时间，还能扩大北京及周边地区企业的海外市场。', '6月30日，第一趟北京到阿塞拜疆巴库的火车出发。', '新华社记者 张晨霖 摄', '北京市房山区商务局局长路鹏介绍，火车的成功开通，意味着北京中欧班列多元通道体系进一步完善，形成北京与欧洲之间“陆上直达+铁海联运”的国际运输布局。', '6月30日，在北京到阿塞拜疆巴库的火车出发前，起重机吊装货箱。', '新华社记者 张晨霖 摄', '记者从当日举办的北京房山创建国家物流枢纽推介会了解到，房山区今年已开行9趟国际货运班列，逐步构建辐射亚欧大陆的物流网络。', '路鹏说，房山将把促进物流建设与当地产业发展深度融合，为高端制造、绿色能源、新材料等“北京智造”产品提供低成本、高效率的物流通道。', '（记者陈钟昊、张晨霖）', '6月30日，北京到阿塞拜疆巴库的火车停在北京市房山区的北京国际陆港等待出发。', '新华社记者 张晨霖 摄', '分享让更多人看到', '人民日报社概况|关于人民网|报社招聘|招聘英才|广告服务|合作加盟|版权服务|数据服务|网站声明|网站律师|信息保护|联系我们', '人民日报违法和不良信息举报电话：010-65363263    举报邮箱：jubao@people.cn', '人民网服务邮箱：kf@peo